In [5]:
import pandas as pd
import numpy as np
import joblib
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
import warnings
warnings.filterwarnings("ignore")

# ---------------------------------------------
# Paths
# ---------------------------------------------
DATASET_PATH = "/content/Dataset Final (2).xlsx"
BEST_MODEL_FILE = "/content/best_model.joblib"
SCALER_FILE = "/content/scaler.joblib"
ENCODER_CITY_FILE = "/content/city_encoder.joblib"
ENCODER_FUEL_FILE = "/content/fuel_encoder.joblib"

# ---------------------------------------------
# Load Dataset
# ---------------------------------------------
df = pd.read_excel(DATASET_PATH)
print("Original Dataset Shape:", df.shape)

# Drop duplicates + missing
df.drop_duplicates(inplace=True)
df.dropna(inplace=True)

# ---------------------------------------------
# Encoding
# ---------------------------------------------
encoder_city = LabelEncoder()
encoder_fuel = LabelEncoder()

df["city"] = encoder_city.fit_transform(df["city"])
df["fuel_type"] = encoder_fuel.fit_transform(df["fuel_type"])

X = df[["Year", "city", "fuel_type"]]
y = df["value"]

# StandardScaler
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Train/Test Split
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42
)

# ---------------------------------------------
# Train Models
# ---------------------------------------------
models = {
    "Linear Regression": LinearRegression(),
    "Decision Tree": DecisionTreeRegressor(random_state=42),
    "Random Forest": RandomForestRegressor(random_state=42),
    "Gradient Boosting": GradientBoostingRegressor(random_state=42),
}

results = []
trained_models = {}

for name, model in models.items():
    print(f"\nTraining {name}...")
    model.fit(X_train, y_train)
    trained_models[name] = model

    y_pred = model.predict(X_test)

    mse = mean_squared_error(y_test, y_pred)
    rmse = np.sqrt(mse)
    r2 = r2_score(y_test, y_pred)
    mape = (np.abs((y_test - y_pred) / y_test).mean()) * 100
    accuracy = 100 - mape

    results.append((name, rmse, r2, accuracy))
    print(f"{name}: RMSE={rmse:.4f}, R2={r2:.4f}, Accuracy={accuracy:.2f}%")

# ---------------------------------------------
# Best Model Selection
# ---------------------------------------------
best_model_info = max(results, key=lambda x: x[3])
best_model_name = best_model_info[0]
best_model = trained_models[best_model_name]

print("\n🚀 Best Model:", best_model_name)
print(f"🎯 Accuracy: {best_model_info[3]:.2f}%")

# ---------------------------------------------
# Store Best Model + Preprocessors
# ---------------------------------------------
joblib.dump(best_model, BEST_MODEL_FILE)
joblib.dump(scaler, SCALER_FILE)
joblib.dump(encoder_city, ENCODER_CITY_FILE)
joblib.dump(encoder_fuel, ENCODER_FUEL_FILE)

print("\nModel & Preprocessing saved successfully ✔")

# ---------------------------------------------
# Price Prediction Function
# ---------------------------------------------
def predict_fuel_price(year, city_name, fuel_type):
    # reload items
    model = joblib.load(BEST_MODEL_FILE)
    scaler = joblib.load(SCALER_FILE)
    encoder_city = joblib.load(ENCODER_CITY_FILE)
    encoder_fuel = joblib.load(ENCODER_FUEL_FILE)

    # validation
    if city_name not in encoder_city.classes_:
        return "❌ Invalid City Name"
    if fuel_type not in encoder_fuel.classes_:
        return "❌ Invalid Fuel Type"

    city_encoded = encoder_city.transform([city_name])[0]
    fuel_encoded = encoder_fuel.transform([fuel_type])[0]

    input_features = [[year, city_encoded, fuel_encoded]]
    input_scaled = scaler.transform(input_features)

    predicted_value = model.predict(input_scaled)[0]

    return f"Predicted Price: ₹{predicted_value:.2f}"

# ---------------------------------------------
# Test User Input
# ---------------------------------------------
print("\n🔍 Try a Prediction Example:")
test_output = predict_fuel_price(2025, "Delhi", "Petrol")
print(test_output)


Original Dataset Shape: (21712, 4)

Training Linear Regression...
Linear Regression: RMSE=6.7065, R2=0.6610, Accuracy=93.17%

Training Decision Tree...
Decision Tree: RMSE=3.8860, R2=0.8862, Accuracy=96.18%

Training Random Forest...
Random Forest: RMSE=3.8838, R2=0.8863, Accuracy=96.18%

Training Gradient Boosting...
Gradient Boosting: RMSE=3.9032, R2=0.8852, Accuracy=96.18%

🚀 Best Model: Random Forest
🎯 Accuracy: 96.18%

Model & Preprocessing saved successfully ✔

🔍 Try a Prediction Example:
Predicted Price: ₹96.00


In [6]:
# ---------------------------------------------
# Interactive User Input Prediction
# ---------------------------------------------
print("\n==== Fuel Price Prediction Tool ====\n")

year = int(input("Enter Year (e.g., 2025): "))

# Show available cities
encoder_city = joblib.load(ENCODER_CITY_FILE)
print("\nAvailable Cities:", list(encoder_city.classes_))
city_name = input("Enter City Name (Exactly as shown): ")

# Show fuel types
encoder_fuel = joblib.load(ENCODER_FUEL_FILE)
print("\nAvailable Fuel Types:", list(encoder_fuel.classes_))
fuel_type = input("Enter Fuel Type (Exactly as shown): ")

result = predict_fuel_price(year, city_name, fuel_type)
print("\n---------------------------------")
print(result)
print("---------------------------------\n")



==== Fuel Price Prediction Tool ====

Enter Year (e.g., 2025): 2025

Available Cities: ['Chennai', 'Delhi', 'Kolkata', 'Mumbai']
Enter City Name (Exactly as shown): Mumbai

Available Fuel Types: ['Diesel', 'Petrol']
Enter Fuel Type (Exactly as shown): Diesel

---------------------------------
Predicted Price: ₹93.26
---------------------------------

